In [31]:
import os

try:
    import google.colab
    os.system('git clone https://github.com/ethanresek/luminal-classifiers 2>/dev/null; cd /content/luminal-classifiers && git pull')
except ImportError:
    pass

In [44]:
import numpy as np
import pandas as pd
import sys
import joblib
import torch.nn.functional as F

from datetime import datetime
from typing import override
from torch import nn, ones, sigmoid, optim
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from skorch import NeuralNetBinaryClassifier
from skorch.callbacks import EarlyStopping
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score

In [33]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Imports OK")

Imports OK


In [34]:
try:
    from google.colab import drive
    sys.path.append('/content/luminal-classifiers')
    from pre_process import preprocess, split_data
    CSV = '/content/luminal-classifiers/data/METABRIC_RNA_Mutation.csv'
    print('Working in Colab')
except ImportError:
    from pre_process import preprocess, split_data
    CSV = 'data/METABRIC_RNA_Mutation.csv'
    print('Working locally')

Working locally


In [35]:
DF = pd.read_csv(CSV, low_memory=False)
Y_OLD_NAME = 'pam50_+_claudin-low_subtype'
KEEP = list(DF.columns[31:520]) + [Y_OLD_NAME]

X, y = preprocess(DF, KEEP)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=RANDOM_SEED)
X_train, X_test, y_train, y_test = X_train.astype('float32'), X_test.astype('float32'), y_train.astype('float32'), y_test.astype('float32')

In [41]:
class SparsityNNBC(NeuralNetBinaryClassifier):
    def __init__(self, **kwargs):
        super(SparsityNNBC, self).__init__(**kwargs)

    @override
    def get_loss(self, y_pred, y_true, X=None, training=False):
        sparsity_effect = self.module_.sparsity * sigmoid(self.module_.gate).sum()
        loss = super(SparsityNNBC, self).get_loss(y_pred, y_true, X, training) + sparsity_effect
        return loss

In [37]:
class MLP(nn.Module):
    def __init__(self, hidden_sizes, dropout, sparsity, input_size=489, output_size=1):
        super(MLP, self).__init__()

        self.sparsity = sparsity
        self.gate = nn.Parameter(ones(input_size))

        sizes = [input_size] + hidden_sizes + [output_size]
        self.linears = nn.ModuleList([
            nn.Linear(sizes[i], sizes[i+1]) for i in range(len(sizes) - 1)
        ])
        self.dropout = nn.Dropout(dropout)
        self.batch_norms = nn.ModuleList([
            nn.BatchNorm1d(h) for h in hidden_sizes
        ])

    def forward(self, x):

        x = x * sigmoid(self.gate)

        for i, l in enumerate(self.linears[:-1]):
            x = l(x)
            x = self.batch_norms[i](x)
            x = F.relu(x)
            x = self.dropout(x)

        x = self.linears[-1](x)

        return x

In [43]:
net = SparsityNNBC(
    module=MLP,
    criterion=nn.BCEWithLogitsLoss,
    optimizer=optim.Adam,
    max_epochs=100,
    callbacks=[EarlyStopping(patience=10)],
    module__hidden_sizes=[128, 64],
    module__dropout=0.3,
    module__input_size=489,
    module__sparsity=1e-3,
    optimizer__weight_decay=1e-3,
    verbose=0
)

p_dist = {
    'module__hidden_sizes': [[256, 128], [128, 64], [64, 32], [128]],
    'module__dropout': [0.1, 0.2, 0.3, 0.4, 0.5],
    'module__sparsity': [1e-4, 1e-3, 5e-3, 1e-2],
    'optimizer__weight_decay': [1e-4, 1e-3, 1e-2],
    'lr': [1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
    'batch_size': [16, 32, 64],
}

In [ ]:
rand_search = RandomizedSearchCV(net, param_distributions=p_dist, scoring='f1', n_iter=100, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED), random_state=RANDOM_SEED)
rand_search.fit(X_train, y_train)
print(rand_search.best_params_)

In [ ]:
final_net = rand_search.best_estimator_

y_pred = final_net.predict(X_test)
y_pred_prob = final_net.predict_proba(X_test)[:, 1]

print('F1:', f1_score(y_test, y_pred))
print('Balanced Accuracy:', balanced_accuracy_score(y_test, y_pred))
print('ROC AUC:', roc_auc_score(y_test, y_pred_prob))


In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
os.makedirs('models', exist_ok=True)

joblib.dump({
    'model': rand_search.best_estimator_,
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test
}, f'models/tuned_MLP_{timestamp}.joblib')